<a href="https://colab.research.google.com/github/oniviev/ml-intro-project-Nivievskyi/blob/main/Chapter_10_Ex_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Consider the RNN fit to the NYSE data in Section 10.9.6. Modify the
code to allow inclusion of the variable day_of_week, and fit the RNN. Compute the test R2.

In [ ]:
!pip install ISLP

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.preprocessing import StandardScaler
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger
from torchmetrics import R2Score
from torch.optim import RMSprop

In [ ]:
from ISLP import load_data
NYSE = load_data('NYSE')
NYSE.columns

Index(['day_of_week', 'DJ_return', 'log_volume', 'log_volatility', 'train'], dtype='object')

Data consisting of the Dow Jones returns, log trading volume, and log volatility for the New York Stock Exchange over a 20 year period

date: Date

day_of_week: Day of the week

DJ_return: Return for Dow Jones Industrial Average

log_volume: Log of trading volume

log_volatility: Log of volatility





In [ ]:
# 1. Load and standardize NYSE data
NYSE = load_data('NYSE')
cols = ['DJ_return', 'log_volume', 'log_volatility']
X_df = NYSE[cols]
X_scaled = StandardScaler(with_mean=True, with_std=True).fit_transform(X_df)
X = pd.DataFrame(X_scaled, columns=cols, index=NYSE.index)

Scaled the features to have a mean of 0 and a standard deviation of 1. This is important for machine learning models like RNNs because it ensures all features are on the same scale.

In [ ]:
# 2. Create lagged features for lags 1 through 5
for lag in range(1, 6):
    for col in cols:
        X[f"{col}_{lag}"] = X[col].shift(lag)
# Include the train indicator and drop NaNs
X['train'] = NYSE['train']
X = X.dropna()

We created lagged features for the DJ_return, log_volume, and log_volatility columns, each with 5 lags, added the train indicator to identify the rows for training. Dropped rows with NaN values that resulted from the lagging operation.



In [ ]:
# 3. Prepare response and training mask, drop current-day columns
Y = X['log_volume']
train_mask = X['train'].astype(bool)
X = X.drop(columns=['train'] + cols)

train_mask) was created to indicate which rows belong to the training set (True for training, False for testing

In [ ]:
# 4. Reorder columns so time runs first -> last across lag dimension
ordered_cols = []
for lag in range(5, 0, -1):
    for col in cols:
        ordered_cols.append(f"{col}_{lag}")
X = X.reindex(columns=ordered_cols)

In this step, the goal was to reorder the columns in X so that the lagged features are arranged in reverse order from the most recent to the least recent. This ensured that the most recent data (lag 1) appears first in the dataset, which is important for time series analysis.

In [ ]:
# 5. Reshape to (n_samples, n_timesteps=5, n_features=3)
X_rnn = X.to_numpy(dtype=np.float32).reshape(-1, 5, 3)

we reshape the data from a 2D format into a 3D tensor suitable for input to the RNN model.

In [ ]:
# 6. Build day_of_week dummies aligned to X.index
dow = pd.get_dummies(NYSE['day_of_week'], prefix='dow').loc[X.index]
dow_np = dow.to_numpy(dtype=np.float32)        # shape (n, 7)
# 7. Tile dummies across timesteps and concatenate
dow_expanded = np.repeat(dow_np[:, None, :], 5, axis=1)  # (n,5,7)
X_rnn_day = np.concatenate([X_rnn, dow_expanded], axis=2) # (n,5,10)

In these steps, we incorporated the day_of_week feature into the data by converting it into one-hot encoded columns, expanding it across the 5 timesteps, and then concatenating it with the reshaped lagged data.

In [ ]:
# 8. Split into train/test TensorDatasets
datasets = []
for mask in [train_mask.values, ~train_mask.values]:
    X_t = torch.tensor(X_rnn_day[mask], dtype=torch.float32)
    Y_t = torch.tensor(Y[mask].to_numpy(), dtype=torch.float32)
    datasets.append(TensorDataset(X_t, Y_t))
nyse_train, nyse_test = datasets

In this step, we splited the data into training and test sets

In [ ]:

# 9. Create a LightningDataModule without ISLP
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader

class NYSEDataModule(LightningDataModule):
    def __init__(self, train_ds, val_ds, test_ds, batch_size=64, num_workers=4):
        super().__init__()
        self.train_ds = train_ds
        self.val_ds   = val_ds
        self.test_ds  = test_ds
        self.batch_size  = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,  num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_ds,   batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_ds,  batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

nyse_dm = NYSEDataModule(
    train_ds=nyse_train,
    val_ds=nyse_test,
    test_ds=nyse_test,
    batch_size=64,
    num_workers=min(4, 6)
)


In this step, we created a custom LightningDataModule for the NYSE dataset

In [ ]:
# 10. Define the LightningModule for RNN regression
from pytorch_lightning import LightningModule

class RNNRegressor(LightningModule):
    def __init__(self, input_size, hidden_size=12, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.rnn     = nn.RNN(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.fc      = nn.Linear(hidden_size, 1)
        self.loss_fn = nn.MSELoss()
        self.r2      = R2Score()

    def forward(self, x):
        out, _   = self.rnn(x)                   # (batch, timesteps, hidden)
        h_last   = out[:, -1, :]                 # final timestep
        h_drop   = self.dropout(h_last)
        return self.fc(h_drop).squeeze(1)        # (batch,)

    def step(self, batch, stage):
        x, y     = batch
        y_hat    = self(x)
        loss     = self.loss_fn(y_hat, y)
        r2_metric= self.r2(y_hat, y)
        self.log(f"{stage}_loss", loss,    prog_bar=True)
        self.log(f"{stage}_r2",    r2_metric, prog_bar=True)
        return loss

    def training_step(self,   batch, batch_idx): return self.step(batch, "train")
    def validation_step(self, batch, batch_idx): return self.step(batch, "val")
    def test_step(self,       batch, batch_idx): return self.step(batch, "test")

    def configure_optimizers(self):
        return RMSprop(self.parameters(), lr=self.hparams.lr)


# Determine the correct input size from concatenated array:
input_dim = X_rnn_day.shape[2]

# Instantiate the model with the dynamic input size
rnn_model = RNNRegressor(input_size=input_dim, hidden_size=12, lr=1e-3)

We defined an RNN regression model using PyTorch Lightning. It uses Mean Squared Error (MSE) as the loss function and R² score as the evaluation metric.

In [ ]:


# 11. Set up Trainer and Logger
logger  = CSVLogger("logs", name="NYSE_RNN_DOW")
trainer = Trainer(
    max_epochs=200,
    deterministic=True,
    logger=logger
)



INFO:pytorch_lightning.utilities.rank_zero:Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
# 12. Train & Evaluate
trainer.fit(rnn_model, datamodule=nyse_dm)
results = trainer.test(rnn_model, datamodule=nyse_dm)

print(f"Test R² with day_of_week in RNN: {results[0]['test_r2']:.4f}")


INFO:pytorch_lightning.callbacks.model_summary:
  | Name    | Type    | Params | Mode 
--------------------------------------------
0 | rnn     | RNN     | 264    | train
1 | dropout | Dropout | 0      | train
2 | fc      | Linear  | 13     | train
3 | loss_fn | MSELoss | 0      | train
4 | r2      | R2Score | 0      | train
--------------------------------------------
277       Trainable params
0         Non-trainable params
277       Total params
0.001     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=200` reached.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.5679774880409241     │
│          test_r2          │    0.3384256064891815     │
└───────────────────────────┴───────────────────────────┘

Test R² with day_of_week in RNN: 0.3384


Test R²: The test_r2 score is 0.33, which means that the model explains about 33% of the variance in the test data, a score of 0.3384 suggests that the model can be improved.